# Price Band Performance ETL

## Purpose
Analyze product performance by price segments (budget, economy, standard, premium, luxury, ultra) to identify pricing sweet spots and optimize pricing strategy.

## Input
* **Source:** `big_data.silver.order_products` 
* **Source:** `big_data.silver.products_enriched` 

## Output
* **Target:** `big_data.gold.vw_price_band_performance`
* **Refresh:** Real-time (always reflects current Silver data)

## SQL Logic
1. JOIN order_products with products_enriched on product_id
2. GROUP BY price_band
3. COUNT orders and calculate reorder rate per band
4. ORDER BY price_band

In [0]:
%sql
-- Price Band Performance View
-- Purpose: Analyze product performance across 6 price segments

CREATE OR REPLACE VIEW big_data.gold.vw_price_band_performance AS
SELECT 
  p.price_band,
  COUNT(*) AS times_ordered,
  ROUND(SUM(CASE WHEN op.reordered THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS reorder_rate
FROM big_data.silver.order_products op
LEFT JOIN big_data.silver.products_enriched p ON op.product_id = p.product_id
GROUP BY p.price_band
ORDER BY p.price_band;

In [0]:
%sql
-- Verify view exists and preview all 6 price bands
-- Returns 6 rows (budget, economy, standard, premium, luxury, ultra)

SELECT * FROM big_data.gold.vw_price_band_performance;